In [ ]:
from pathlib import Path
import csv

GT_DIR   = Path(r'C:\Users\arnau\projetde\runs-in-behind\GT')
PRED_DIR = Path(r'C:\Users\arnau\projetde\runs-in-behind\outputs_loop_with_offsets')

gt_files   = sorted(GT_DIR.glob("*.csv"))
pred_files = sorted(PRED_DIR.glob("*.csv"))


# ── helpers ───────────────────────────────────────────────────────────────────

def f1(precision, recall):
    """Harmonic mean of precision and recall. Returns 0 if both are 0."""
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def evaluate(gt_data, pred_data):
    """Return (TP, FP, FN) given a list of GT start-frames
    and a list of predicted (start_frame, end_frame) tuples."""
    TP = FN = 0
    for gt_start in gt_data:
        hit = any(p_start <= gt_start <= p_end for p_start, p_end in pred_data)
        if hit:
            TP += 1
        else:
            FN += 1
    FP = len(pred_data) - TP
    return TP, FP, FN


def read_gt(filepath):
    with open(filepath, newline='') as f:
        return [int(row["start_frame"]) for row in csv.DictReader(f)]


def read_pred(filepath):
    with open(filepath, newline='') as f:
        return [(int(row["start_frame"]), int(row["end_frame"]))
                for row in csv.DictReader(f)]


# ── per-match evaluation ──────────────────────────────────────────────────────

def main():
    results = []

    SEP = "─" * 60
    print(SEP)
    print(f"{'Match':<14} {'GT':>4} {'Pred':>5} {'TP':>4} {'FP':>4} {'FN':>4}  "
          f"{'Recall':>7}  {'Prec':>7}  {'F1':>7}")
    print(SEP)

    for gt_file, pred_file in zip(gt_files, pred_files):
        match_name = gt_file.stem

        gt_data   = read_gt(gt_file)
        pred_data = read_pred(pred_file)

        TP, FP, FN = evaluate(gt_data, pred_data)

        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        score_f1  = f1(precision, recall)

        results.append(dict(
            match=match_name,
            gt=len(gt_data), pred=len(pred_data),
            TP=TP, FP=FP, FN=FN,
            recall=recall, precision=precision, f1=score_f1
        ))

        print(f"{match_name:<14} {len(gt_data):>4} {len(pred_data):>5} "
              f"{TP:>4} {FP:>4} {FN:>4}  "
              f"{recall:>7.2%}  {precision:>7.2%}  {score_f1:>7.2%}")

    # ── global (micro) summary ────────────────────────────────────────────────
    print(SEP)

    total_TP   = sum(r['TP']   for r in results)
    total_FP   = sum(r['FP']   for r in results)
    total_FN   = sum(r['FN']   for r in results)
    total_gt   = sum(r['gt']   for r in results)
    total_pred = sum(r['pred'] for r in results)

    global_recall    = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0.0
    global_precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0.0
    global_f1        = f1(global_precision, global_recall)

    print(f"{'GLOBAL':<14} {total_gt:>4} {total_pred:>5} "
          f"{total_TP:>4} {total_FP:>4} {total_FN:>4}  "
          f"{global_recall:>7.2%}  {global_precision:>7.2%}  {global_f1:>7.2%}")
    print(SEP)

    # ── macro averages ────────────────────────────────────────────────────────
    n = len(results)
    macro_recall    = sum(r['recall']    for r in results) / n
    macro_precision = sum(r['precision'] for r in results) / n
    macro_f1        = sum(r['f1']        for r in results) / n

    print(f"\nMacro avg (moyenne sur {n} matchs)")
    print(f"  Recall    : {macro_recall:.2%}")
    print(f"  Precision : {macro_precision:.2%}")
    print(f"  F1        : {macro_f1:.2%}")

    # ── best / worst match ────────────────────────────────────────────────────
    best  = max(results, key=lambda r: r['f1'])
    worst = min(results, key=lambda r: r['f1'])
    print(f"\nMeilleur match  : {best['match']}  (F1 = {best['f1']:.2%})")
    print(f"Moins bon match : {worst['match']}  (F1 = {worst['f1']:.2%})")


if __name__ == "__main__":
    main()